# 04. Explainability and Fairness Analysis

This notebook covers model interpretability and fairness evaluation:
- **Permutation Importance** on test dataset
- Decision threshold tuning (0.10 to 0.90)
- **Partial Dependence Plot (PDP)** & **ICE** for top features
- **LIME** local predictions
- Subgroup recall and fairness metrics (**Demographic Parity**, **Equalized Odds**, **Disparate Impact**)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import confusion_matrix, recall_score
from imblearn.over_sampling import SMOTE

sns.set(style="whitegrid")

# Data Pipeline Reconstruction
data = pd.read_csv("../data/creditcard.csv")
df = data.copy()
df["Log_Amount"] = np.log1p(df["Amount"])
df["Is_High_Amount"] = (df["Amount"] >= 100).astype(int)

X = df.drop("Class", axis=1)
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled, X_test_scaled = X_train.copy(), X_test.copy()
for col in ["Time", "Amount", "Log_Amount"]:
    X_train_scaled[col] = scaler.fit_transform(X_train[[col]])
    X_test_scaled[col] = scaler.transform(X_test[[col]])

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X_train_res, y_train_res)
selector = SelectFromModel(rf_temp, threshold="median", prefit=True)
X_train_sel = selector.transform(X_train_res)
X_test_sel = selector.transform(X_test_scaled)
selected_features = X_train_scaled.columns[selector.get_support()]
X_test_sel_df = pd.DataFrame(X_test_sel, columns=selected_features)

rf = joblib.load("../models/random_forest.pkl")
y_proba_rf = rf.predict_proba(X_test_sel)[:, 1]
y_pred_rf = (y_proba_rf >= 0.5).astype(int)

## 1. Permutation Importance & Threshold Tuning

In [ ]:
def plot_permutation_importance(rf, X_test_sel, y_test, selected_features):
    perm = permutation_importance(rf, X_test_sel, y_test, n_repeats=10, random_state=42)
    perm_df = pd.DataFrame({"feature": selected_features, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(x="importance", y="feature", data=perm_df.head(10))
    plt.title("Permutation Importance - Random Forest")
    plt.show()

def threshold_tuning(y_proba_rf, y_test):
    print("\n=== THRESHOLD TUNING ===")
    thresholds = np.linspace(0.1, 0.9, 9)
    for t in thresholds:
        y_pred_t = (y_proba_rf >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
        recall_t = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
        print(f"Threshold {t:.2f} | Precision {precision_t:.4f} | Recall {recall_t:.4f}")

plot_permutation_importance(rf, X_test_sel, y_test, selected_features)
threshold_tuning(y_proba_rf, y_test)

## 2. Partial Dependence (PDP/ICE) and Local Explanations (LIME)

In [ ]:
def pdp_ice_plot(rf, X_test_sel_df, selected_features):
    feature_for_pdp = "V14" if "V14" in selected_features else selected_features[0]
    PartialDependenceDisplay.from_estimator(rf, X_test_sel_df, [feature_for_pdp], kind="both")
    plt.title(f"PDP/ICE for {feature_for_pdp}")
    plt.show()

def lime_explanations(X_train_sel, X_test_sel, selected_features, y_test, rf):
    try:
        from lime.lime_tabular import LimeTabularExplainer
        explainer = LimeTabularExplainer(training_data=X_train_sel, feature_names=list(selected_features), class_names=["Non-Fraud", "Fraud"], mode="classification", discretize_continuous=True)
        fraud_indices = np.where(y_test == 1)[0]
        for idx in fraud_indices[:5]:
            exp = explainer.explain_instance(data_row=X_test_sel[idx], predict_fn=rf.predict_proba, num_features=10)
            print(f"\n=== LIME Explanation for Test Index {idx} (Fraud Case) ===")
            print(exp.as_list())
    except ImportError:
        print("LIME package not installed.")

pdp_ice_plot(rf, X_test_sel_df, selected_features)
lime_explanations(X_train_sel, X_test_sel, selected_features, y_test, rf)

## 3. Subgroup Recall & Algorithmic Fairness

In [ ]:
def recall_by_amount_group(X_test_scaled, y_test, y_pred_rf):
    median_amt = X_test_scaled["Amount"].median()
    high_group = X_test_scaled["Amount"] > median_amt
    recall_high = recall_score(y_test[high_group], y_pred_rf[high_group])
    recall_low = recall_score(y_test[~high_group], y_pred_rf[~high_group])
    print("\n=== Recall by Amount Group ===")
    print("High Amount Recall:", recall_high)
    print("Low Amount Recall:", recall_low)

def fairness_metrics(y_test, y_pred_rf, X_test_scaled):
    median_amt = X_test_scaled["Amount"].median()
    high_group = X_test_scaled["Amount"] > median_amt
    y_true_high, y_pred_high = y_test[high_group], y_pred_rf[high_group]
    y_true_low, y_pred_low = y_test[~high_group], y_pred_rf[~high_group]

    def rates_from_confusion(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        pos_rate = (tp + fp) / (tp + fp + tn + fn)
        return tpr, fpr, pos_rate

    tpr_h, fpr_h, pos_rate_h = rates_from_confusion(y_true_high, y_pred_high)
    tpr_l, fpr_l, pos_rate_l = rates_from_confusion(y_true_low, y_pred_low)

    print("\n=== Fairness Metrics (High vs Low Amount Groups) ===")
    print(f"High Amount - TPR (Recall): {tpr_h:.4f}, FPR: {fpr_h:.4f}, Positive Rate: {pos_rate_h:.4f}")
    print(f"Low Amount  - TPR (Recall): {tpr_l:.4f}, FPR: {fpr_l:.4f}, Positive Rate: {pos_rate_l:.4f}")

    dp_ratio = pos_rate_h / pos_rate_l if pos_rate_l > 0 else np.nan
    print(f"Demographic Parity Ratio (High / Low): {dp_ratio:.4f}")
    print(f"Equalized Odds - |TPR High - TPR Low|: {abs(tpr_h - tpr_l):.4f}")
    print(f"Equalized Odds - |FPR High - FPR Low|: {abs(fpr_h - fpr_l):.4f}")
    print(f"Disparate Impact (High vs Low): {dp_ratio:.4f}")

recall_by_amount_group(X_test_scaled, y_test, y_pred_rf)
fairness_metrics(y_test, y_pred_rf, X_test_scaled)